In [ ]:
!pip install -q transformers timm accelerate bitsandbytes scikit-learn underthesea

In [ ]:
import os
import ast
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
from torchvision import models
from transformers import Blip2Processor, Blip2Model, AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.preprocessing import LabelEncoder
import warnings
from torch.cuda.amp import autocast, GradScaler
from underthesea import word_tokenize

warnings.filterwarnings("ignore")

# Đưa đường dẫn train_box_csv lên CONFIG
CONFIG = {
    "train_csv": "/kaggle/input/datasets/hydrogenhydrogen/vivqa-1/ViVQA-main/ViVQA-main/train.csv",
    "val_csv": "/kaggle/input/datasets/hydrogenhydrogen/vivqa-1/ViVQA-main/ViVQA-main/test.csv",
    "train_img_dir": "/kaggle/input/datasets/hydrogenhydrogen/vivqa-1/train",
    "val_img_dir": "/kaggle/input/datasets/hydrogenhydrogen/vivqa-1/test",
    "train_box_csv": "/kaggle/input/datasets/hydrogenhydrogen/vivqa-1/merged_train.csv", 
    
    "save_path": "/kaggle/working/multitask_box_vivqa.pth",
    "label_encoder_path": "label_encoder.pkl",
    
    "blip_model": "Salesforce/blip2-opt-2.7b", 
    "text_model": "vinai/phobert-base",      
    
    "batch_size": 32,
    "epochs": 25,
    "lr": 5e-5,
    "weight_decay": 0.05,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    
    "type_embed_dim": 64,
    "max_length": 50,
    "patience": 5,
    "delta": 0.001,

    # Weighted per-sample answer loss theo q_type (boost type 1)
    "type_loss_weights": {0: 4.0, 1: 0.5, 2: 4.0, 3: 4.0}
}

def preprocess_vietnamese(text):
    text = str(text).lower().strip()
    tokens = word_tokenize(text, format="text") 
    return tokens

def find_image_path(folder, img_id):
    valid_exts = ['.jpg', '.png', '.jpeg']
    img_id_str = str(img_id)
    if img_id_str.lower().endswith(tuple(valid_exts)):
        path = os.path.join(folder, img_id_str)
        if os.path.exists(path): return path
    for ext in valid_exts:
        path = os.path.join(folder, f"{img_id_str}{ext}")
        if os.path.exists(path): return path
    return None

class ViVQADataset(Dataset):
    def __init__(self, dataframe, image_dir, blip_processor, tokenizer, label_encoder, unk_token_id):
        self.data = dataframe
        self.image_dir = image_dir
        self.blip_processor = blip_processor
        self.tokenizer = tokenizer
        self.label_encoder = label_encoder
        self.unk_token_id = unk_token_id
        self.known_classes = set(label_encoder.classes_)

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        
        # 1. XỬ LÝ ẢNH
        img_id = row.get('img_id', row.get('image'))
        img_path = find_image_path(self.image_dir, img_id)
        try:
            image = Image.open(img_path).convert("RGB") if img_path else Image.new('RGB', (224, 224))
        except:
            image = Image.new('RGB', (224, 224))
            
        orig_w, orig_h = image.size 
        pixel_values = self.blip_processor(images=image, return_tensors="pt")['pixel_values'].squeeze(0)

        # 2. XỬ LÝ TỌA ĐỘ BOX (TARGET CHO MULTI-TASK)
        target_box = torch.zeros(4, dtype=torch.float32)
        has_box = torch.tensor(0.0, dtype=torch.float32)
        
        # Sử dụng get để an toàn nếu cột owl_box không tồn tại (tập Val)
        box_str = row.get('merged_box', None)
        if pd.notna(box_str):
            try:
                box = ast.literal_eval(str(box_str))
                xmin, ymin, xmax, ymax = box
                
                target_box[0] = xmin / orig_w
                target_box[1] = ymin / orig_h
                target_box[2] = xmax / orig_w
                target_box[3] = ymax / orig_h
                
                target_box = torch.clamp(target_box, 0.0, 1.0)
                has_box = torch.tensor(1.0, dtype=torch.float32)
            except:
                pass

        # 3. XỬ LÝ TEXT & LABEL
        question = preprocess_vietnamese(row['question'])
        text_encoding = self.tokenizer(
            question, return_tensors="pt", padding="max_length", 
            truncation=True, max_length=CONFIG['max_length'], add_special_tokens=True
        )

        answer = str(row['answer']).lower().strip()
        label = self.label_encoder.transform([answer])[0] if answer in self.known_classes else self.unk_token_id
            
        return {
            "pixel_values": pixel_values,
            "input_ids": text_encoding['input_ids'].squeeze(0),
            "attention_mask": text_encoding['attention_mask'].squeeze(0),
            "q_type": torch.tensor(int(row.get('type', 0)), dtype=torch.long),
            "labels": torch.tensor(label, dtype=torch.long),
            "target_box": target_box, 
            "has_box": has_box        
        }

class CrossAttentionFusion(nn.Module):
    def __init__(self, visual_dim, text_dim, embed_dim, num_heads=8):
        super().__init__()
        self.vis_project = nn.Linear(visual_dim, embed_dim)
        self.txt_project = nn.Linear(text_dim, embed_dim)
        self.multihead_attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True, dropout=0.1)
        self.norm = nn.LayerNorm(embed_dim)
        
    def forward(self, visual_features, text_features):
        v_proj = self.vis_project(visual_features)
        t_proj = self.txt_project(text_features)
        
        attn_output, _ = self.multihead_attn(query=t_proj, key=v_proj, value=v_proj)
        combined = self.norm(attn_output + t_proj)
        
        avg_pool = combined.mean(dim=1)
        max_pool = combined.max(dim=1)[0]
        
        return avg_pool + max_pool

class PhoBERT_BLIP2_EfficientNet_MultiTask(nn.Module):
    def __init__(self, num_classes, num_q_types):
        super().__init__()
        
        # --- BLIP-2 ---
        print("⏳ Đang load BLIP-2...")
        tmp_blip = Blip2Model.from_pretrained(CONFIG['blip_model'], torch_dtype=torch.float16)
        self.vision_model = tmp_blip.vision_model
        self.qformer = tmp_blip.qformer
        self.query_tokens = tmp_blip.query_tokens 
        del tmp_blip 
        
        for param in self.vision_model.parameters(): param.requires_grad = False
        for param in self.qformer.parameters(): param.requires_grad = False
        self.query_tokens.requires_grad = False
            
        # --- EfficientNet-B7 ---
        print("⏳ Đang load EfficientNet-B7...")
        try:
            weights = models.EfficientNet_B7_Weights.DEFAULT
            self.efficientnet = models.efficientnet_b7(weights=weights)
        except:
            self.efficientnet = models.efficientnet_b7(pretrained=True)
            
        self.eff_feature_extractor = self.efficientnet.features
        for param in self.efficientnet.parameters(): param.requires_grad = False
        self.local_proj = nn.Linear(2560, 768)
        
        # --- PhoBERT ---
        print("⏳ Đang load PhoBERT...")
        self.phobert = AutoModel.from_pretrained(CONFIG['text_model'])
        
        # --- Fusion ---
        embed_dim = 768
        self.fusion = CrossAttentionFusion(visual_dim=768, text_dim=768, embed_dim=embed_dim)
        self.type_emb = nn.Embedding(num_q_types + 1, CONFIG['type_embed_dim'])
        
        in_features = embed_dim + CONFIG['type_embed_dim']
        
        self.classifier = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(512, num_classes)
        )
        
        self.bbox_head = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 4),
            nn.Sigmoid() 
        )

    def forward(self, pixel_values, input_ids, attention_mask, q_type):
        with torch.no_grad():
            vision_outputs = self.vision_model(pixel_values=pixel_values)
            image_embeds = vision_outputs.last_hidden_state
            
            image_attention_mask = torch.ones(image_embeds.size()[:-1], dtype=torch.long, device=image_embeds.device)
            query_tokens = self.query_tokens.expand(image_embeds.shape[0], -1, -1)
            
            query_outputs = self.qformer(
                query_embeds=query_tokens,
                encoder_hidden_states=image_embeds,
                encoder_attention_mask=image_attention_mask,
                output_attentions=False 
            )
            global_feats = query_outputs.last_hidden_state 

        with torch.no_grad():
            local_maps = self.eff_feature_extractor(pixel_values) 
        B, C, H, W = local_maps.shape
        local_feats = local_maps.view(B, C, -1).permute(0, 2, 1) 
        local_feats = self.local_proj(local_feats) 
        
        visual_feats = torch.cat([global_feats, local_feats], dim=1)
        text_outputs = self.phobert(input_ids=input_ids, attention_mask=attention_mask)
        text_feats = text_outputs.last_hidden_state
        visual_feats = visual_feats.to(text_feats.dtype)
        
        fused_vec = self.fusion(visual_feats, text_feats)
        type_vec = self.type_emb(q_type)
        
        final_features = torch.cat([fused_vec, type_vec], dim=-1)
        
        logits = self.classifier(final_features)
        pred_boxes = self.bbox_head(final_features)
        
        return logits, pred_boxes

class EarlyStopping:
    def __init__(self, patience=5, delta=0, path='checkpoint.pth'):
        self.patience, self.delta, self.path = patience, delta, path
        self.counter, self.best_score, self.early_stop = 0, None, False

    def __call__(self, val_acc, model):
        if self.best_score is None:
            self.best_score = val_acc
            self.save_checkpoint(model)
        elif val_acc < self.best_score + self.delta:
            self.counter += 1
            if self.counter >= self.patience: self.early_stop = True
        else:
            self.best_score = val_acc
            self.save_checkpoint(model)
            self.counter = 0
    
    def save_checkpoint(self, model):
        torch.save(model.state_dict(), self.path)

def train():
    blip_processor = Blip2Processor.from_pretrained(CONFIG['blip_model'])
    tokenizer = AutoTokenizer.from_pretrained(CONFIG['text_model'], use_fast=False)
    
    train_df = pd.read_csv(CONFIG['train_csv'])
    val_df = pd.read_csv(CONFIG['val_csv'])
    
    print("⏳ Đang chuẩn bị dữ liệu...")
    
    # Chỉ load Box cho tập Train từ CONFIG
    df_box_train = pd.read_csv(CONFIG['train_box_csv'])
    
    # Chuẩn hóa tên cột question_id
    for df in [train_df, val_df, df_box_train]:
        if 'Unnamed: 0' in df.columns:
            df.rename(columns={'Unnamed: 0': 'question_id'}, inplace=True)
        elif df.columns[0] == 'Unnamed: 0' or 'question_id' not in df.columns:
            df.columns = ['question_id'] + list(df.columns[1:])
            
    # Gộp dữ liệu Box vào train_df. val_df sẽ không có cột owl_box
    train_df = pd.merge(train_df, df_box_train[['question_id', 'owl_box']], on='question_id', how='left')
            
    train_answers = train_df['answer'].apply(lambda x: str(x).lower().strip()).unique().tolist()
    if 'unknown' not in train_answers: train_answers.append('unknown')
    
    label_encoder = LabelEncoder().fit(train_answers)
    unk_token_id = label_encoder.transform(['unknown'])[0]
    
    train_loader = DataLoader(
        ViVQADataset(train_df, CONFIG['train_img_dir'], blip_processor, tokenizer, label_encoder, unk_token_id), 
        batch_size=CONFIG['batch_size'], shuffle=True, num_workers=0, pin_memory=True
    )
    val_loader = DataLoader(
        ViVQADataset(val_df, CONFIG['val_img_dir'], blip_processor, tokenizer, label_encoder, unk_token_id), 
        batch_size=CONFIG['batch_size'], shuffle=False, num_workers=0, pin_memory=True
    )
    
    num_q_types = int(max(train_df['type'].max(), val_df['type'].max()) + 1)
    model = PhoBERT_BLIP2_EfficientNet_MultiTask(len(label_encoder.classes_), num_q_types).to(CONFIG['device'])
    
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * len(train_loader) * CONFIG['epochs']), num_training_steps=len(train_loader) * CONFIG['epochs'])
    
    # Weighted per-sample VQA loss (reduction='none')
    criterion_vqa = nn.CrossEntropyLoss(reduction='none', label_smoothing=0.1) 
    criterion_bbox = nn.SmoothL1Loss(reduction='none') 
    
    # Tạo weight lookup tensor trên GPU
    type_weight_tensor = torch.ones(num_q_types, device=CONFIG['device'])
    for t, w in CONFIG['type_loss_weights'].items():
        if t < num_q_types:
            type_weight_tensor[t] = w
    
    scaler = GradScaler()
    early_stopping = EarlyStopping(patience=5, path=CONFIG['save_path'])

    LAMBDA_BBOX = 2.0 

    print(f"Type loss weights: {CONFIG['type_loss_weights']}")
    for epoch in range(CONFIG['epochs']):
        model.train()
        train_correct, train_total = 0, 0
        # Theo dõi accuracy theo từng q_type
        train_type_correct = {t: 0 for t in range(num_q_types)}
        train_type_total = {t: 0 for t in range(num_q_types)}
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
        
        for batch in pbar:
            pixel_values = batch['pixel_values'].to(CONFIG['device'], dtype=torch.float16)
            input_ids = batch['input_ids'].to(CONFIG['device'])
            attention_mask = batch['attention_mask'].to(CONFIG['device'])
            q_type = batch['q_type'].to(CONFIG['device'])
            labels = batch['labels'].to(CONFIG['device'])
            
            target_box = batch['target_box'].to(CONFIG['device']) 
            has_box = batch['has_box'].to(CONFIG['device'])       
            
            optimizer.zero_grad()
            with autocast():
                logits, pred_boxes = model(pixel_values, input_ids, attention_mask, q_type)
                
                # Weighted per-sample VQA loss theo q_type
                per_sample_loss = criterion_vqa(logits, labels)           # [batch_size]
                sample_weights = type_weight_tensor[q_type]               # [batch_size]
                loss_vqa = (per_sample_loss * sample_weights).mean()
                
                loss_bbox_raw = criterion_bbox(pred_boxes, target_box).mean(dim=1) 
                # has_box = 0 đối với mẫu không có nhãn box, giúp loại bỏ nhiễu
                loss_bbox = (loss_bbox_raw * has_box).sum() / (has_box.sum() + 1e-6)
                
                loss = loss_vqa + (LAMBDA_BBOX * loss_bbox)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            
            preds = logits.argmax(1)
            train_correct += (preds == labels).sum().item()
            train_total += labels.size(0)
            
            # Thống kê accuracy theo từng q_type
            for t in range(num_q_types):
                mask = (q_type == t)
                if mask.sum() > 0:
                    train_type_correct[t] += (preds[mask] == labels[mask]).sum().item()
                    train_type_total[t] += mask.sum().item()
            
            pbar.set_postfix({
                'acc': f'{train_correct/train_total:.4f}',
                'L_vqa': f'{loss_vqa.item():.3f}',
                'L_box': f'{loss_bbox.item():.3f}'
            })

        # --- VALIDATION ---
        model.eval()
        val_correct, val_total = 0, 0
        val_type_correct = {t: 0 for t in range(num_q_types)}
        val_type_total = {t: 0 for t in range(num_q_types)}
        
        with torch.no_grad(), autocast():
            for batch in val_loader:
                pv = batch['pixel_values'].to(CONFIG['device'], dtype=torch.float16)
                ii = batch['input_ids'].to(CONFIG['device'])
                am = batch['attention_mask'].to(CONFIG['device'])
                qt = batch['q_type'].to(CONFIG['device'])
                lb = batch['labels'].to(CONFIG['device'])
                
                logits, _ = model(pv, ii, am, qt)
                preds = logits.argmax(1)
                val_correct += (preds == lb).sum().item()
                val_total += lb.size(0)
                
                for t in range(num_q_types):
                    mask = (qt == t)
                    if mask.sum() > 0:
                        val_type_correct[t] += (preds[mask] == lb[mask]).sum().item()
                        val_type_total[t] += mask.sum().item()
        
        val_acc = val_correct / val_total
        print(f"Epoch {epoch+1} | Train Acc: {train_correct/train_total:.4f} | Val Acc: {val_acc:.4f}")
        
        # In accuracy từng q_type
        print(f"    --- Per Q-Type Answer Accuracy ---")
        for t in range(num_q_types):
            tr_acc = train_type_correct[t] / train_type_total[t] if train_type_total[t] > 0 else 0
            vl_acc = val_type_correct[t] / val_type_total[t] if val_type_total[t] > 0 else 0
            w = CONFIG['type_loss_weights'].get(t, 1.0)
            marker = " <<<" if w > 1.0 else ""
            print(f"    Type {t} (w={w}): Train={tr_acc:.4f} | Val={vl_acc:.4f}{marker}")
        
        early_stopping(val_acc, model)
        if early_stopping.early_stop:
            print("Early stopping triggered")
            break

if __name__ == '__main__':
    train()

In [ ]:
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from PIL import Image
import os
import gc
from sklearn.preprocessing import LabelEncoder
from transformers import Blip2Processor, AutoTokenizer

# ==========================================
# 1. CẤU HÌNH INFERENCE
# ==========================================
INFERENCE_CONFIG = {
    "train_csv": CONFIG['train_csv'], # Dùng để rebuild LabelEncoder & num_q_types
    "test_csv": CONFIG['val_csv'],    # File test/val cần chạy inference
    "img_dir": CONFIG['val_img_dir'],
    "model_path": CONFIG['save_path'],
    "output_file": "ket_qua_test_multitask.csv",
    "batch_size": 16, 
    "device": CONFIG['device']
}

# ==========================================
# 2. DATASET CHO INFERENCE
# ==========================================
class ViVQAMultiTaskInferenceDataset(Dataset):
    def __init__(self, dataframe, image_dir, blip_processor, tokenizer):
        self.data = dataframe
        self.image_dir = image_dir
        self.blip_processor = blip_processor
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        
        img_id = str(row.get('img_id', row.get('image')))
        question_text = str(row['question'])
        gt_answer = str(row['answer']) if 'answer' in row else ""
        q_type = int(row.get('type', 0))
        
        img_path = find_image_path(self.image_dir, img_id)
        try:
            image = Image.open(img_path).convert("RGB") if img_path else Image.new('RGB', (224, 224))
        except:
            image = Image.new('RGB', (224, 224))
            
        orig_w, orig_h = image.size # LƯU KÍCH THƯỚC GỐC ĐỂ KHÔI PHỤC BOX
        pixel_values = self.blip_processor(images=image, return_tensors="pt")['pixel_values'].squeeze(0)

        processed_q = preprocess_vietnamese(question_text)
        text_encoding = self.tokenizer(
            processed_q, 
            return_tensors="pt", 
            padding="max_length", 
            truncation=True, 
            max_length=CONFIG['max_length'],
            add_special_tokens=True
        )
        
        return {
            "pixel_values": pixel_values,
            "input_ids": text_encoding['input_ids'].squeeze(0),
            "attention_mask": text_encoding['attention_mask'].squeeze(0),
            "q_type": torch.tensor(q_type, dtype=torch.long),
            
            # Meta data để xuất file CSV
            "meta_img_id": img_id,
            "meta_question": question_text,
            "meta_gt_answer": gt_answer,
            "meta_orig_w": orig_w,
            "meta_orig_h": orig_h
        }

def run_inference_and_save():
    # --- Dọn dẹp bộ nhớ ---
    print(">>> Đang dọn dẹp bộ nhớ GPU...")
    if 'model' in globals(): del globals()['model']
    gc.collect()
    torch.cuda.empty_cache()

    print(">>> Đang chuẩn bị dữ liệu cho Inference Multi-task...")
    
    # 1. Rebuild LabelEncoder (Khớp với tập Train)
    train_df = pd.read_csv(INFERENCE_CONFIG['train_csv'])
    train_answers = train_df['answer'].apply(lambda x: str(x).lower().strip()).unique().tolist()
    if 'unknown' not in train_answers: train_answers.append('unknown')
    
    label_encoder = LabelEncoder().fit(train_answers)
    num_classes = len(label_encoder.classes_)
    
    # Lấy num_q_types để khởi tạo model
    test_df = pd.read_csv(INFERENCE_CONFIG['test_csv'])
    num_q_types = int(max(train_df['type'].max(), test_df['type'].max()) + 1)

    # 2. Khởi tạo model Đa nhiệm
    print(">>> Khởi tạo model PhoBERT_BLIP2_EfficientNet_MultiTask...")
    model = PhoBERT_BLIP2_EfficientNet_MultiTask(num_classes, num_q_types)
    
    # 3. Load Weights
    if os.path.exists(INFERENCE_CONFIG['model_path']):
        print(f"Loading weights từ: {INFERENCE_CONFIG['model_path']}")
        state_dict = torch.load(INFERENCE_CONFIG['model_path'], map_location='cpu')
        model.load_state_dict(state_dict)
    else:
        print("!!! LỖI: Không tìm thấy file checkpoint. Hãy chắc chắn model đã train xong.")
        return

    model = model.to(INFERENCE_CONFIG['device'])
    model.eval()

    # 4. Chuẩn bị DataLoader Test
    blip_processor = Blip2Processor.from_pretrained(CONFIG['blip_model'])
    tokenizer = AutoTokenizer.from_pretrained(CONFIG['text_model'], use_fast=False)
    
    test_dataset = ViVQAMultiTaskInferenceDataset(test_df, INFERENCE_CONFIG['img_dir'], blip_processor, tokenizer)
    test_loader = DataLoader(test_dataset, batch_size=INFERENCE_CONFIG['batch_size'], shuffle=False, num_workers=2)
    
    results = []
    print(">>> Bắt đầu chạy dự đoán...")
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Testing"):
            pixel_values = batch['pixel_values'].to(INFERENCE_CONFIG['device'], dtype=torch.float16)
            input_ids = batch['input_ids'].to(INFERENCE_CONFIG['device'])
            attention_mask = batch['attention_mask'].to(INFERENCE_CONFIG['device'])
            q_type = batch['q_type'].to(INFERENCE_CONFIG['device'])
            
            with torch.cuda.amp.autocast(enabled=(INFERENCE_CONFIG['device']=="cuda")):
                # Đa nhiệm: Lấy cả 2 output
                ans_logits, pred_boxes = model(pixel_values, input_ids, attention_mask, q_type)
            
            # --- Xử lý Kết quả VQA ---
            pred_ans_indices = ans_logits.argmax(dim=1).cpu().numpy()
            pred_ans_text = label_encoder.inverse_transform(pred_ans_indices)
            
            # --- Xử lý Kết quả Bounding Box ---
            pred_boxes_np = pred_boxes.cpu().numpy() # Shape: (Batch, 4)
            
            batch_size = input_ids.size(0)
            for i in range(batch_size):
                # Khôi phục box chuẩn hóa [0, 1] về kích thước pixel ảnh gốc
                w = batch['meta_orig_w'][i].item()
                h = batch['meta_orig_h'][i].item()
                box_norm = pred_boxes_np[i]
                
                real_box = [
                    int(box_norm[0] * w), # x_min
                    int(box_norm[1] * h), # y_min
                    int(box_norm[2] * w), # x_max
                    int(box_norm[3] * h)  # y_max
                ]
                
                results.append({
                    "ID ảnh": batch['meta_img_id'][i],
                    "Câu hỏi": batch['meta_question'][i],
                    "Đáp án (GT)": batch['meta_gt_answer'][i],
                    "Đáp án Dự đoán": pred_ans_text[i],
                    "BBox Dự đoán (x1, y1, x2, y2)": str(real_box)
                })

    # 5. Lưu kết quả
    result_df = pd.DataFrame(results)
    cols = ["ID ảnh", "Câu hỏi", "Đáp án (GT)", "Đáp án Dự đoán", "BBox Dự đoán (x1, y1, x2, y2)"]
    result_df = result_df[cols]
    
    result_df.to_csv(INFERENCE_CONFIG['output_file'], index=False, encoding='utf-8-sig')
    print(f"\n>>> Đã lưu kết quả thành công tại: {INFERENCE_CONFIG['output_file']}")
    
    # Tính Accuracy nếu file test có chứa đáp án thực tế
    if result_df["Đáp án (GT)"].iloc[0] != "":
        acc = (result_df["Đáp án (GT)"].astype(str).str.lower().str.strip() == \
               result_df["Đáp án Dự đoán"].astype(str).str.lower().str.strip()).mean()
        print(f"Độ chính xác Test VQA: {acc:.4f}")

if __name__ == "__main__":
    run_inference_and_save()